In [2]:
pip install langchain


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
imports utils.util
import streamlit as st

user_upload = st.file_uploader("파일을 업로드",accept_multipie_files=False)
if user_upload is not None:
    if st.button("Upload"):
        with st.spinner("PDF 처리중.."):
            #PDF 텍스트 가져오기
            raw_text = util.get_pdf_text(user_upload)
            util.send_api(raw_text,"pdf")

SyntaxError: invalid syntax (258429357.py, line 1)

In [12]:
import gradio as gr

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama


from dotenv import load_dotenv

load_dotenv()

# PDF 파일 로드, 파일의 경로 입력
loader = PyPDFLoader("./AI브리프_3월_260303.pdf")

docs = loader.load() # pdf 한 페이지 별로 document 객체로 변환되서 반환됨.


nomic_embeddings = OllamaEmbeddings(model="nomic-embed-text") # ollama에서 기본적으로 제공해주는 모델
bge_embeddings = OllamaEmbeddings(model="bge-m3") # 한국어 포함한 다국어로 학습이 된 모델

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

text_chunk = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(text_chunk, bge_embeddings)

retriever = vectorstore.as_retriever(search_kwargs={'k':2}) # 검색 문서는 최대 2개의 문서만 이용하도록 설정


# RAG Chain - Gemma2 활용
# 한국어 성능이 비교적 좋음

# 답변 속도가 오래걸리기 때문에 groq 이용


llm = ChatGroq(
    model = "llama-3.1-8b-instant",
    temperature = 0.2,
    max_retries = 2,
)

# Prompt
template = ''' Answer the question based only on the following context.

[Context]
{context}

[Question]
{question}

[Answer (in Korean)]
'''

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])


# RAG Chain 연결
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 채팅 함수
def chat(message, history):

    try:
        response = rag_chain.invoke(message)
        return response

    except Exception as e:
        return f"에러 발생: {str(e)}"

# Gradio Chat UI
demo = gr.ChatInterface(
    fn=chat,
    title="📚 RAG 챗봇",
    description="문서를 기반으로 답변하는 AI 챗봇입니다.",
    textbox=gr.Textbox(
        placeholder="질문을 입력하세요...",
        container=True,
        scale=7
    ),
)

# 실행
demo.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.
